In [1]:
%%html
<style type="text/css">
  span.ecb { background: yellow; }
</style>
<span class="ecb">Grading comments will be in yellow</span>

In [2]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy
from hw5_eqs import *

# ATMO 5322 - Homework #4
### Spring 2026, Due 5 May, 2026, 2 pm (before class).

This homework builds on Homework 4 to calculate 

For homework 4 and 5, you my work individually or with a group of up to two other class members. If you work together, please say who you worked with in the assignment and only turn in one assignment on GitHub.

When working these problems, you will need to code up equations for and make plots of various electrostatic quantities. Turn in a Jupyter notebook with your figures and which I can run to reproduce your results. Label your axes like a professional.

Please also enter your derivations into the notebook using $\LaTeX$. Provide any needed discussion as commentary in the notebook in a Markdown cell.

To the extent possible, move supporting calculation code to another file and import it so the notebook is shorter and cleaner.

The next set of questions will scale up the calculation of electric field change and its inversion using the code from the previous part of this assignment. This will allow us to study the errors in retrieving charges and their locations starting from known input data.

### Q1: Generating dE from known monopolar discharges

To study the sensitivity of solution quality to its position within the field change network, we will next move the monopolar total charge amount from stroke 6a to many different locations. Use a 1 km grid covering (-10, 10) in x and (-15, 15) in y, and assume the charge is lowered from 3, 5, and 7 km above ground. Predict the field change at each station (you don't need to plot it).

This calculation can be vectorized for all grid locations and one site. In my implementation the calculation of dE for all grid boxes for all eight sites runs almost instantly.

### Q2: Inferring the monopolar discharge from the generated dE

You now have dE at all station locations, for all of the (assumed) monopolar discharges. Retreive the locations and charge amounts, pretending you didn't know them. This calculation will be slower, but only required a few seconds to completely run in my implementation.

Calculate and plot the error in the source location and charge magnitude at each position (a total of four error variables). Also plot the location of the measurement statios. There should be twelve plots - sets of four at each altitude. Is there any spatial variaibility in these error measures?

### Q3. Effect of measurement error on retrievals

Add a zero-mean distribution of [Gaussian errors](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.normal.html#numpy.random.Generator.normal) to the measured field changes. Use a standard deviation (aka RMS error) of 150 V/m, independently distributed at each station. 

Repeat the retrieval process and plot diagnostics for each error corrupted measurement. 

### Q4. Statistics of retrieval errors

Repeat Q3 for 30 error-corrupted E field measurements, so as to produce a large sample of statistics of the errors.

Calculate and plot the bias and standard deviation of position (each coordinate) and charge magnitude at each charge source location. This will be 24 plots, using all 30 realizations of the retrieval at each grid point.

### Q5: Visualizing the error stats as a chi^2 distribution
Calculate the reduced chi-squared value (in V/m) associated with each retrieved source, accounting for the degrees of freedom $\nu$, and normalizing by the RMS error you added above. You will have about 54000 values = (20x30x3 spatial)x(30 retrievals, one for each realization of error at each station). Create a histogram of these chi-squared values, and compare to the [analytic chi-squared distribution](https://en.wikipedia.org/wiki/Chi-squared_distribution) for that $\nu$, as in one of the panels in Thomas et al. (2004), Figure 9.

### Q1: Generating dE from known monopolar discharges

To study the sensitivity of solution quality to its position within the field change network, we will next move the monopolar total charge amount from stroke 6a to many different locations. Use a 1 km grid covering (-10, 10) in x and (-15, 15) in y, and assume the charge is lowered from 3, 5, and 7 km above ground. Predict the field change at each station (you don't need to plot it).

This calculation can be vectorized for all grid locations and one site. In my implementation the calculation of dE for all grid boxes for all eight sites runs almost instantly.

In [3]:
# station df from sam
station_df = pd.DataFrame([
    {'name' : 'STROZZI', 'x' : -200, 'y' : 100, 'z' : 0},
    {'name' : 'RECEIVING VANS', 'x' : -2750, 'y' : -4100, 'z' : 0},
    {'name' : 'WATER CANYON', 'x' : 4100, 'y' : -5000, 'z' : 0},
    {'name' : 'KELLY', 'x' : 4500, 'y' : -200, 'z' : 0},
    {'name' : 'GUTIERREZ', 'x' : -4000, 'y' : 3000, 'z' : 0},
    {'name' : 'ABANDONED HOUSE', 'x' : -800, 'y' : 5000, 'z' : 0},
    {'name' : 'BOONDOCK', 'x' : 4500, 'y' : 8500, 'z' : 0},
    {'name' : 'WINDMILL', 'x' : 900, 'y' : 10000, 'z' : 0},
    {'name' : 'OUTOFSIGHT', 'x' : -4800, 'y' : 12500, 'z' : 0},
    # {'name' : '3 CM RADAR', 'x' : 0, 'y' : 0, 'z' : 1830},
    # {'name' : 'ETOT1', 'x' : 0, 'y' : 5100, 'z' : 1830},
    # {'name' : 'ETOT2', 'x' : 2900, 'y' : 400, 'z' : 1830},
    # {'name' : 'ETOT3', 'x' : 3100, 'y' : -6400, 'z' : 1830},
])

station_df

,name,x,y,z
0,STROZZI,-200,100,0
1,RECEIVING VANS,-2750,-4100,0
2,WATER CANYON,4100,-5000,0
3,KELLY,4500,-200,0
4,GUTIERREZ,-4000,3000,0
5,ABANDONED HOUSE,-800,5000,0
6,BOONDOCK,4500,8500,0
7,WINDMILL,900,10000,0
8,OUTOFSIGHT,-4800,12500,0


In [ ]:
# # create the arrays where the charge is located
# Q = 3.7 # C

# xgrid_q = np.arange(-10e3,11e3, 1e3) # in meters
# ygrid_q = np.arange(-15e3, 16e3, 1e3)
# z_q = np.array([3e3, 5e3, 7e3])

# X, Y, Z = np.meshgrid(xgrid_q, ygrid_q, z_q)

# Ri_dict = {}
# mono_dE = {}

# # calculate Ri vector for each station
# for name in station_df['name']:
#     x_obs = station_df.loc[station_df['name'] == name]['x'].values[0]
#     y_obs = station_df.loc[station_df['name'] == name]['y'].values[0]
    
#     Ri_x = X - x_obs
#     Ri_y = Y - y_obs
#     Ri_z = Z

#     Ri_mag = np.sqrt(Ri_x**2 + Ri_y**2 + Ri_z**2)

#     dE = monopole_dE(Q, Ri_mag, Ri_z) # dE at each station


#     mono_dE[name] = {'dE (kV/m)': dE/1000}

#     Ri_dict[name] = {
#         'Ri_x': Ri_x,
#         'Ri_y': Ri_y,
#         'Ri_z': Ri_z,
#         'Ri_mag': Ri_mag
#     }

In [4]:
charge_df = pd.read_csv('hw4_charge.csv')
unique_types = {name : group for name, group in charge_df.groupby('change_type')}
monopole_df =  unique_types['monopole']
dipole_df = unique_types['pointdipole']
monopole_df

,stroke_num,q,qx,qy,qz,drx,dry,drz,change_type
1,1.1,3.9,-500.0,4200.0,4000.0,NaN,NaN,NaN,monopole
2,2.0,1.0,100.0,7100.0,5500.0,NaN,NaN,NaN,monopole
3,3.0,2.8,-50.0,6000.0,4700.0,NaN,NaN,NaN,monopole
4,4.0,2.3,-100.0,5100.0,4600.0,NaN,NaN,NaN,monopole
5,5.0,6.7,-500.0,4900.0,5200.0,NaN,NaN,NaN,monopole
6,6.1,3.7,-1500.0,2200.0,4600.0,NaN,NaN,NaN,monopole
7,6.2,2.2,-1000.0,1200.0,4900.0,NaN,NaN,NaN,monopole
8,6.3,6.0,-1700.0,500.0,4900.0,NaN,NaN,NaN,monopole


In [9]:
qx = np.array([-1500] * 3)
qy = np.array([2200] * 3)
qz = np.array([3000, 5000, 7000])

Q = 3.7
d = {'q':Q, 'qx':qx, 'qy':qy, 'qz':qz}
stroke_df = pd.DataFrame(data=d)

In [10]:
stroke_df

,q,qx,qy,qz
0,3.7,-1500,2200,3000
1,3.7,-1500,2200,5000
2,3.7,-1500,2200,7000


In [29]:
rows = []
rows_R = []
    
for name in station_df['name']:
    for i in range(len(stroke_df['qz'])):
        x_obs = station_df.loc[station_df['name'] == name]['x'].values[0]
        y_obs = station_df.loc[station_df['name'] == name]['y'].values[0]
        
        Ri_x = qx[i] - x_obs
        Ri_y = qy[i] - y_obs
        Ri_z = qz[i]
    
        Ri_mag = np.sqrt(Ri_x**2 + Ri_y**2 + Ri_z**2)
    
        dE = monopole_dE(Q, Ri_mag, Ri_z) # dE at each station
    
        row = {
        'station': name,
        'dE (kV/m)': dE / 1000,
        'qx':qx[i],
        'qy':qy[i],
        'qz': qz[i],
            
        }
    
        rows.append(row)
    
        Ri_rows = {
            'Ri_x': Ri_x,
            'Ri_y': Ri_y,
            'Ri_z': Ri_z,
            'Ri_mag': Ri_mag,
            'qz': qz[i]
        }
        rows_R.append(Ri_rows)

mono_dE = pd.DataFrame(rows)
Ri = pd.DataFrame(rows_R)

In [30]:
mono_dE

,station,dE (kV/m),qx,qy,qz
0,STROZZI,3.400464,-1500,2200,3000
1,STROZZI,1.917396,-1500,2200,5000
2,STROZZI,1.138290,-1500,2200,7000
3,RECEIVING VANS,0.560102,-1500,2200,3000
4,RECEIVING VANS,0.616664,-1500,2200,5000
5,RECEIVING VANS,0.542990,-1500,2200,7000
6,WATER CANYON,0.225376,-1500,2200,3000
7,WATER CANYON,0.295469,-1500,2200,5000
8,WATER CANYON,0.306290,-1500,2200,7000
9,KELLY,0.551723,-1500,2200,3000


In [25]:
Ri

,Ri_x,Ri_y,Ri_z,Ri_mag,qz
0,-1300,2100,3000,3885.871846,3000
1,-1300,2100,5000,5576.737397,5000
2,-1300,2100,7000,7422.937424,7000
3,1250,6300,3000,7088.899774,3000
4,1250,6300,5000,8139.563870,5000
5,1250,6300,7000,9500.131578,7000
6,-5600,7200,3000,9602.083107,3000
7,-5600,7200,5000,10401.922899,5000
8,-5600,7200,7000,11497.825881,7000
9,-6000,2400,3000,7124.605252,3000


### Q2: Inferring the monopolar discharge from the generated dE

You now have dE at all station locations, for all of the (assumed) monopolar discharges. Retreive the locations and charge amounts, pretending you didn't know them. This calculation will be slower, but only required a few seconds to completely run in my implementation.

Calculate and plot the error in the source location and charge magnitude at each position (a total of four error variables). Also plot the location of the measurement statios. There should be twelve plots - sets of four at each altitude. Is there any spatial variaibility in these error measures?

In [55]:
max_obs_idx = np.argmax(np.abs(mono_dE['dE (kV/m)']))

guess_x = mono_dE['qx'][max_obs_idx]
guess_y = mono_dE['qy'][max_obs_idx]
guess_z = mono_dE['qz'][max_obs_idx]

guess_q = 1 # C

lsq_rows = []

for i in stroke_df['qz'].values:
    mono_dE_z = mono_dE[mono_dE['qz']==i]
    stations_subset = station_df.set_index('name').loc[mono_dE_z['station']].reset_index()
    lsq_get, _ = scipy.optimize.leastsq(delta_E_error, x0=np.array([guess_q, guess_x, guess_y, guess_z]), args=(mono_dE_z['dE (kV/m)'].values, 
                                                                                                                stations_subset))
    print(f'Retrieved parameters for stroke 5.0: q={lsq_get[0]:.2f} C, x={lsq_get[1]:.2f} m, y={lsq_get[2]:.2f} m, z={lsq_get[3]:.2f} m')

    row = {
            'lsq_q':lsq_get[0].round(1),
            'lsq_x':lsq_get[1].round(1),
            'lsq_y':lsq_get[2].round(1),
            'lsq_z':lsq_get[3].round(1)
          }

    lsq_rows.append(row)

lsq_results = pd.DataFrame(lsq_rows)

Retrieved parameters for stroke 5.0: q=3.70 C, x=-1500.00 m, y=2200.00 m, z=3000.00 m
Retrieved parameters for stroke 5.0: q=3.70 C, x=-1500.00 m, y=2200.00 m, z=5000.00 m
Retrieved parameters for stroke 5.0: q=3.70 C, x=-1500.00 m, y=2200.00 m, z=7000.00 m


In [56]:
lsq_results

,lsq_q,lsq_x,lsq_y,lsq_z
0,3.7,-1500.0,2200.0,3000.0
1,3.7,-1500.0,2200.0,5000.0
2,3.7,-1500.0,2200.0,7000.0


#### messed up

In [ ]:
# # make the dE flat (combine x and y for each z) for leastsq

# Xf = X.ravel()
# Yf = Y.ravel()
# Zf = Z.ravel()

# n_points = Xf.size

In [ ]:
# n_stations = len(station_df)

# dE_all = np.zeros((n_points, n_stations))

# for i, name in enumerate(station_df['name']):
#     dE_all[:, i] = mono_dE[name]['dE (kV/m)'].ravel()

In [ ]:
# dE_all.shape

In [ ]:
# dE_all_df = pd.DataFrame(data=dE_all)
# dE_all_df = dE_all_df.T
# dE_all_df['name'] = station_df['name']
# dE_all_df

In [ ]:
# which station had the largest overall dE reading? need for best guess. do the sum of all dE per station

total_dE = np.sum(np.abs(dE_all), axis=1)
largest_obs_index = np.argmax(total_dE)
guess_Xf = Xf[largest_obs_index]
guess_Yf = Yf[largest_obs_index]
guess_Zf = Zf[largest_obs_index]
guess_q = 1 # C

In [ ]:
total_dE

In [ ]:
flat_idx = np.argmax(np.abs(dE_all))
point_idx, station_idx = np.unravel_index(flat_idx, dE_all.shape)
guess_Xf = Xf[point_idx]
guess_Yf = Yf[point_idx]
guess_Zf = Zf[point_idx]
guess_Xf

In [ ]:
least_sq_get, _ = scipy.optimize.leastsq(delta_E_error, x0 = np.array([guess_q, guess_Xf, guess_Yf, guess_Zf]), args=(dE_all_df, station_df[station_df['name'] == dE_all_df['name']]))
print(f'Retrieved parameters for stroke 5.0: q={least_sq_get[0]:.2f} C, x={least_sq_get[1]:.2f} m, y={least_sq_get[2]:.2f} m, z={least_sq_get[3]:.2f} m')

In [ ]:
def delta_E_error(params, observed_E, station_info):
    q, x, y, z = params
    predicted_E = monopole_dE(x, y, z, station_info['x'], station_info['y'], q)
    return observed_E - np.reshape(predicted_E, observed_E.shape)

In [ ]:
# make the initial guess the location of the station with the largest observed change in E for that stroke
stroke_5_obs = all_stroke_df['stroke_5.0']
largest_obs_index = np.argmax(np.abs(stroke_5_obs))
initial_guess_q = 1 # C
initial_guess_x = station_df['x'][largest_obs_index]
initial_guess_y = station_df['y'][largest_obs_index]
initial_guess_z = station_df['z'][largest_obs_index]


retrieved_5_opt, _ = leastsq(delta_E_error,
                   x0=np.array([initial_guess_q, initial_guess_x, initial_guess_y, initial_guess_z]), # initial guess
                   args=(stroke_5_obs, # all station observations for stroke 5.0
                         station_df[station_df['name'] == all_stroke_df['station']])) # make sure the station list is sorted the same was as in the all_stroke_df
print(f'Retrieved parameters for stroke 5.0: q={retrieved_5_opt[0]:.2f} C, x={retrieved_5_opt[1]:.2f} m, y={retrieved_5_opt[2]:.2f} m, z={retrieved_5_opt[3]:.2f} m')

### Q3. Effect of measurement error on retrievals

Add a zero-mean distribution of [Gaussian errors](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.normal.html#numpy.random.Generator.normal) to the measured field changes. Use a standard deviation (aka RMS error) of 150 V/m, independently distributed at each station. 

Repeat the retrieval process and plot diagnostics for each error corrupted measurement. 

### Q4. Statistics of retrieval errors

Repeat Q3 for 30 error-corrupted E field measurements, so as to produce a large sample of statistics of the errors.

Calculate and plot the bias and standard deviation of position (each coordinate) and charge magnitude at each charge source location. This will be 24 plots, using all 30 realizations of the retrieval at each grid point.

### Q5: Visualizing the error stats as a chi^2 distribution
Calculate the reduced chi-squared value (in V/m) associated with each retrieved source, accounting for the degrees of freedom $\nu$, and normalizing by the RMS error you added above. You will have about 54000 values = (20x30x3 spatial)x(30 retrievals, one for each realization of error at each station). Create a histogram of these chi-squared values, and compare to the [analytic chi-squared distribution](https://en.wikipedia.org/wiki/Chi-squared_distribution) for that $\nu$, as in one of the panels in Thomas et al. (2004), Figure 9.